In [38]:
import numpy as np
import pandas as pd
import matplotlib as plt

In [39]:
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

train.head()

,ID,text,label
0,1,Atât e de înfricoșat de gândești că e turbat a...,graiul moldovenesc
1,3,Cei ce n-au dreptate n-ar mai năzui în veci la...,graiul moldovenesc
2,5,Deoarece ați primit bani de la oaspetele dumne...,graiul moldovenesc
3,7,Primiți vă rog oameni buni această mică mulțăm...,graiul moldovenesc
4,9,Fiin'c-o vint la noi vezi binie baș în zâua dă...,graiul bănățean


In [40]:
comb = pd.concat([train, test], ignore_index=True)

output = pd.DataFrame(columns=["subtaskID", "datapointID", "answer"])

output.loc[0] = [1, 1, comb["text"].str.count(r"\bpâni\b").sum()]

comb.head()

,ID,text,label
0,1,Atât e de înfricoșat de gândești că e turbat a...,graiul moldovenesc
1,3,Cei ce n-au dreptate n-ar mai năzui în veci la...,graiul moldovenesc
2,5,Deoarece ați primit bani de la oaspetele dumne...,graiul moldovenesc
3,7,Primiți vă rog oameni buni această mică mulțăm...,graiul moldovenesc
4,9,Fiin'c-o vint la noi vezi binie baș în zâua dă...,graiul bănățean


In [41]:
md = train[train["label"] == "graiul moldovenesc"]
bn = train[train["label"] == "graiul bănățean"]

ptmd = md["text"].str.count(r"[^\w\s]")
ptbn = bn["text"].str.count(r"[^\w\s]")

output.loc[1] = [2, 1, round(abs(ptmd.mean() - ptbn.mean()), 2)]

In [42]:
counts = test["text"].str.count(r"[ăâîșțĂÂÎȘȚ]")

st3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": test["ID"],
    "answer": counts
})

In [43]:
import nltk
from pathlib import Path
import re
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

nltk_data_dir = Path("nltk_data")
nltk_data_dir.mkdir(exist_ok=True)
nltk.data.path.append(str(nltk_data_dir.resolve()))
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", download_dir=str(nltk_data_dir), quiet=True, raise_on_error=True)

stop_words = set(stopwords.words('romanian'))
stemmer = SnowballStemmer('romanian')
tokenizer = RegexpTokenizer(r'\w+')

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    tokens = tokenizer.tokenize(text)
    # tokens = [stemmer.stem(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

X = train["text"].astype(str)
y = train["label"]

model = make_pipeline(
    TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 6),
        min_df=1,
        sublinear_tf=True,
        lowercase=True,
    ),
    LogisticRegression(
        C=2,
        class_weight="balanced",
        max_iter=5000,
        random_state=42,
    ),
)

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cv_macro_f1 = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")

# print(f"Macro F1: {cv_macro_f1.mean():.4f} ± {cv_macro_f1.std():.4f}")
# print(cv_macro_f1)

model.fit(X, y)

st4 = pd.DataFrame({
    "subtaskID": 4,
    "datapointID": test["ID"],
    "answer": model.predict(test["text"].astype(str))
})

In [44]:
output = pd.concat([output, st3, st4], ignore_index=True)
output["subtaskID"] = output["subtaskID"].astype(int)
output["datapointID"] = output["datapointID"].astype(int)

output.to_csv("output.csv", index=False)